# ECCT on LDPC(49,24)

This notebook clones the repository, checks the Kaggle GPU, trains ECCT on the same LDPC code and model dimensions as AECCT, and prints the resulting BER/FER log.

Enable **Internet** and a **GPU accelerator** in Kaggle before running. This fair bounded-runtime comparison uses 500 ECCT epochs with 200 batches per epoch, matching AECCT's 250 epochs per phase, 200 batches per epoch, across its two phases.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/gouravanirudh05/SRIP_LDPC_Decoding_using_Machine_Learning.git'
REPO_DIR = Path('/kaggle/working/ldpc_repo')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

ECCT_DIR = REPO_DIR / 'ECCT'
if not (ECCT_DIR / 'Main.py').exists():
    raise FileNotFoundError('ECCT/Main.py is missing from the cloned repository.')

os.chdir(ECCT_DIR)
print('Working directory:', Path.cwd())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'tqdm'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. In Kaggle, select GPU under Notebook options.')
print('GPU:', torch.cuda.get_device_name(0))

## Training configuration

The code, rate, optimizer, batch size, seed, number of blocks, and embedding dimension match the AECCT run. ECCT uses 500 epochs and 200 batches per epoch, giving 100,000 optimizer updates, equal to AECCT's two 250-epoch phases.

In [ ]:
import time

ECCT_EPOCHS = 500
command = [
    sys.executable, 'Main.py',
    '--gpus=0',
    f'--epochs={ECCT_EPOCHS}',
    '--workers=4',
    '--lr=1e-4',
    '--batch_size=128',
    '--train_batches_per_epoch=200',
    '--test_batch_size=2048',
    '--seed=42',
    '--code_type=LDPC',
    '--code_n=49',
    '--code_k=24',
    '--N_dec=6',
    '--d_model=128',
    '--h=8',
]
print('Running:', ' '.join(command))
start = time.perf_counter()
subprocess.run(command, cwd=str(ECCT_DIR), check=True)
print(f'Total wall time: {(time.perf_counter() - start) / 3600:.2f} hours')

In [ ]:
# Print the latest ECCT result directory and its final log lines.
result_dirs = sorted((ECCT_DIR / 'Results_ECCT').glob('*'), key=lambda p: p.stat().st_mtime)
if not result_dirs:
    raise FileNotFoundError('No ECCT result directory was produced.')
latest = result_dirs[-1]
log_file = latest / 'logging.txt'
print('Result directory:', latest)
print('Checkpoint:', latest / 'best_model')
print('\n'.join(log_file.read_text(errors='replace').splitlines()[-40:]))

In [ ]:
from pathlib import Path
import tarfile

# For ECCT:
result_root = Path("/kaggle/working/ldpc_repo/ECCT/Results_ECCT")

# For AECCT, use instead:
# result_root = Path("/kaggle/working/ldpc_repo/AECCT-main/logs/Results_AECCT")

result_dirs = sorted(result_root.glob("*"), key=lambda p: p.stat().st_mtime)
latest = result_dirs[-1]

archive = Path("/kaggle/working/LDPC49_results.tar.gz")

with tarfile.open(archive, "w:gz") as tar:
    tar.add(latest, arcname=latest.name)

print("Saved:", archive)
print("Log:", latest / "logging.txt")
print("Checkpoint:", latest / "best_model")